# Step 1: technical features per symbol and timeframe

Builds a feature table from OHLC bars: stochastic oscillator, EMA trend, RSI, ATR, Bollinger position and recent returns, computed separately for every symbol and timeframe. Indicators are written in plain pandas so there is no compiled dependency.

In [ ]:
import os
from pathlib import Path

# All inputs and outputs live under DATA_DIR (relative to the repo by default).
DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))
OUT_DIR = DATA_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

In [ ]:
import numpy as np
import pandas as pd


def synthetic_ohlc(symbols=("AAA", "BBB", "CCC"), timeframes=("H1", "H4", "D1"), n=3000, seed=SEED):
    rng = np.random.default_rng(seed)
    freq = {"H1": "h", "H4": "4h", "D1": "D"}
    frames = []
    for s in symbols:
        for tf in timeframes:
            idx = pd.date_range("2019-01-01", periods=n, freq=freq[tf])
            close = 1.0 + 0.1 * rng.random() + np.cumsum(rng.normal(0, 0.002, n))
            open_ = np.r_[close[0], close[:-1]]
            high = np.maximum(open_, close) + rng.uniform(0, 0.002, n)
            low = np.minimum(open_, close) - rng.uniform(0, 0.002, n)
            frames.append(pd.DataFrame(dict(symbol=s, timeframe=tf, date=idx, open=open_, high=high,
                                            low=low, close=close, volume=rng.integers(100, 5000, n))))
    return pd.concat(frames, ignore_index=True)

In [ ]:
def ema(s, n):
    return s.ewm(span=n, adjust=False).mean()


def rsi(close, n=14):
    d = close.diff()
    up, down = d.clip(lower=0).ewm(alpha=1 / n, adjust=False).mean(), (-d.clip(upper=0)).ewm(alpha=1 / n, adjust=False).mean()
    return 100 - 100 / (1 + up / down)


def add_features(g):
    g = g.sort_values("date").copy()
    lo, hi = g.low.rolling(20).min(), g.high.rolling(20).max()
    g["stoch_k"] = 100 * (g.close - lo) / (hi - lo)
    g["stoch_d"] = g.stoch_k.rolling(6).mean()
    g["stoch_up"] = (g.stoch_k > g.stoch_d).astype(float)
    g["ema_20"], g["ema_200"] = ema(g.close, 20), ema(g.close, 200)
    g["ema_trend_up"] = (g.ema_20 > g.ema_200).astype(float)
    g["dist_ema_200"] = g.close / g.ema_200 - 1
    g["rsi_14"] = rsi(g.close)
    prev = g.close.shift()
    tr = pd.concat([g.high - g.low, (g.high - prev).abs(), (g.low - prev).abs()], axis=1).max(axis=1)
    g["atr_14_pct"] = tr.rolling(14).mean() / g.close
    mid, sd = g.close.rolling(20).mean(), g.close.rolling(20).std()
    g["bb_position"] = (g.close - mid) / (2 * sd)
    for k in (1, 5, 20):
        g[f"ret_{k}"] = g.close.pct_change(k)
    return g


raw = synthetic_ohlc()
features = pd.concat([add_features(g) for _, g in raw.groupby(["symbol", "timeframe"])], ignore_index=True).dropna()
features.to_csv(OUT_DIR / "features.csv", index=False)
features.groupby(["symbol", "timeframe"]).size()